In [1]:
import sqlite3

# This connects to your database file and creates the 'cursor'
conn = sqlite3.connect('student_database.db')
cursor = conn.cursor()

In [2]:
cursor.execute("DROP TABLE IF EXISTS students")
cursor.execute("CREATE TABLE students (id INTEGER PRIMARY KEY AUTOINCREMENT, full_name TEXT, course TEXT, year_level INTEGER)")

clean_data = [
    ("Anna Reyes", "BS ECE", 4),
    ("Mika Santos", "BS CE", 2),
    ("Cassandra G. Fernandez", "BS ECE", 1)
]
cursor.executemany("INSERT INTO students (full_name, course, year_level) VALUES (?, ?, ?)", clean_data)

cursor.execute("DROP TABLE IF EXISTS courses")
cursor.execute("CREATE TABLE courses (course_code TEXT PRIMARY KEY, description TEXT)")
cursor.execute("INSERT INTO courses VALUES ('BS ECE', 'Electronics and Communications Engineering'), ('BS CE', 'Civil Engineering')")
conn.commit()

In [3]:
# 1. Highest year level
cursor.execute("SELECT full_name, course FROM students WHERE year_level = (SELECT MAX(year_level) FROM students)")
print(cursor.fetchall())

[('Anna Reyes', 'BS ECE')]


In [4]:
# 2. Engineering courses
cursor.execute("SELECT * FROM students WHERE course IN (SELECT course_code FROM courses WHERE description LIKE '%Engineering%')")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 4), (2, 'Mika Santos', 'BS CE', 2), (3, 'Cassandra G. Fernandez', 'BS ECE', 1)]


In [5]:
# 3. Courses with multiple students
cursor.execute("SELECT course, COUNT(*) AS total FROM students GROUP BY course HAVING total > 1")
print(cursor.fetchall())

[('BS ECE', 2)]


In [6]:
# 4. Graduation status classification
cursor.execute("SELECT *, CASE WHEN year_level >= 4 THEN 'Graduating' ELSE 'Regular' END AS status FROM students")
print(cursor.fetchall())

[(1, 'Anna Reyes', 'BS ECE', 4, 'Graduating'), (2, 'Mika Santos', 'BS CE', 2, 'Regular'), (3, 'Cassandra G. Fernandez', 'BS ECE', 1, 'Regular')]


In [7]:
# 5. LEFT JOIN observation
cursor.execute("SELECT s.full_name, c.description FROM students s LEFT JOIN courses c ON s.course = c.course_code")
print(cursor.fetchall())

[('Anna Reyes', 'Electronics and Communications Engineering'), ('Mika Santos', 'Civil Engineering'), ('Cassandra G. Fernandez', 'Electronics and Communications Engineering')]


In [ ]:
# 6. Count of three-part names (e.g., Cassie)
cursor.execute("SELECT COUNT(*) FROM students WHERE full_name LIKE '% % %'")
print(cursor.fetchall())

[(1,)]


In [9]:
# 7. Max year per course sorted
cursor.execute("SELECT course, MAX(year_level) FROM students GROUP BY course ORDER BY MAX(year_level) DESC")
print(cursor.fetchall())

[('BS ECE', 4), ('BS CE', 2)]


In [10]:
# 8. Every second record
cursor.execute("SELECT * FROM students WHERE rowid % 2 = 0")
print(cursor.fetchall())

[(2, 'Mika Santos', 'BS CE', 2)]


In [11]:
# 9. Pattern matching (GLOB)
cursor.execute("SELECT full_name FROM students WHERE full_name GLOB '*e*e*'")
print(cursor.fetchall())

[('Anna Reyes',), ('Cassandra G. Fernandez',)]


In [12]:
# 10. Valid course code check
cursor.execute("SELECT full_name, course FROM students WHERE NOT EXISTS (SELECT 1 FROM courses WHERE course_code = students.course)")
print(cursor.fetchall())

[]


In [13]:
# 11. Statistical summary
cursor.execute("SELECT course, COUNT(*), AVG(year_level) FROM students GROUP BY course")
print(cursor.fetchall())

[('BS CE', 1, 2.0), ('BS ECE', 2, 2.5)]


In [14]:
# 12. Name initials
cursor.execute("SELECT full_name, SUBSTR(full_name, 1, 1) AS initial FROM students")
print(cursor.fetchall())

[('Anna Reyes', 'A'), ('Mika Santos', 'M'), ('Cassandra G. Fernandez', 'C')]


In [15]:
# 13. Keyword search 
keyword = input("Enter search keyword: ")
cursor.execute("SELECT * FROM students WHERE full_name LIKE ? LIMIT 1", ('%' + keyword + '%',))
print(cursor.fetchall())

[(3, 'Cassandra G. Fernandez', 'BS ECE', 1)]


In [16]:
# 14. Course with most enrollment
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course ORDER BY COUNT(*) DESC LIMIT 1")
print(cursor.fetchall())

[('BS ECE', 2)]


In [17]:
# 15. Unique courses filtered by name and year
cursor.execute("SELECT DISTINCT course FROM students WHERE full_name LIKE '%a%' AND year_level BETWEEN 1 AND 4")
print(cursor.fetchall())

[('BS ECE',), ('BS CE',)]


In [18]:
# 16. Longest name 
cursor.execute("SELECT * FROM students WHERE LENGTH(full_name) = (SELECT MAX(LENGTH(full_name)) FROM students)")
print(cursor.fetchall())

[(3, 'Cassandra G. Fernandez', 'BS ECE', 1)]


In [19]:
# 17. Min year count for BS CE
cursor.execute("SELECT COUNT(*) FROM students WHERE course = 'BS CE' AND year_level = (SELECT MIN(year_level) FROM students WHERE course = 'BS CE')")
print(cursor.fetchall())

[(1,)]


In [20]:
# 18. Computer-related JOIN search
cursor.execute("SELECT s.full_name, s.year_level, c.description FROM students s JOIN courses c ON s.course = c.course_code WHERE c.description LIKE '%Computer%'")
print(cursor.fetchall())

[]


In [21]:
# 19. Error handling
try:
    cursor.execute("SELECT * FROM unknown_table")
    print(cursor.fetchall())
except Exception as e:
    print(e)

no such table: unknown_table


In [22]:
# 20. Maximum enrollment tie check
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course HAVING COUNT(*) = (SELECT MAX(cnt) FROM (SELECT COUNT(*) AS cnt FROM students GROUP BY course))")
print(cursor.fetchall())

[('BS ECE', 2)]
